In [ ]:
# 1. Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import torch
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler
from darts.models import NHiTSModel, DLinearModel, TiDEModel

# 2. Configuration

FREQUENCY = 'Daily'
HORIZON = 14

RANDOM_STATE = 42

INPUT_CHUNK_LENGTH = 28   
OUTPUT_CHUNK_LENGTH = HORIZON

N_EPOCHS = 20             
BATCH_SIZE = 256

TRAIN_SERIES_SAMPLE = 1000
MAX_SAMPLES_PER_TS = 20

ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"

data_path = Path('../data/M4')
train_file = data_path / f'{FREQUENCY}-train.csv'
test_file = data_path / f'{FREQUENCY}-test.csv'

print(f"Accelerator: {ACCELERATOR}")

# 3. Metrics
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (metrica ufficiale M4)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)

def mase(y_true, y_pred, y_train, seasonality=1):
    """Mean Absolute Scaled Error"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)
    mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_train[seasonality:] - y_train[:-seasonality]))
    if naive_mae == 0:
        return np.nan
    return mae / naive_mae

print("Metrics defined: sMAPE, MASE")

# 4. Load Datas

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
print(f"Series: {len(train_df)}")

id_to_idx = {sid: i for i, sid in enumerate(train_df.iloc[:, 0])}

# 5. Building TimeSeries

all_series = []
series_ids_order = []
for idx in tqdm(range(len(train_df)), desc="Building TimeSeries"):
    series_id = train_df.iloc[idx, 0]
    values = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
    all_series.append(TimeSeries.from_values(values))
    series_ids_order.append(series_id)

scaler = Scaler(StandardScaler(), global_fit=False)
all_series_scaled = scaler.fit_transform(all_series)

print(f"Costruite {len(all_series)} TimeSeries, scaling per-serie applicato.")

rng = np.random.RandomState(RANDOM_STATE)
train_sample_idx = rng.choice(len(all_series_scaled), size=TRAIN_SERIES_SAMPLE, replace=False)
train_series_sample = [all_series_scaled[i] for i in train_sample_idx]

print(f"Serie usate per il training: {len(train_series_sample)}")

# 6. Models Training
common_kwargs = dict(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    random_state=RANDOM_STATE,
    pl_trainer_kwargs={
        "accelerator": ACCELERATOR,
        "devices": 1,
        "enable_progress_bar": True,
        "enable_checkpointing": False,
        "logger": False,
    },
)

models = {
    "NHiTS": NHiTSModel(**common_kwargs),
    "DLinear": DLinearModel(**common_kwargs),
    "TiDE": TiDEModel(**common_kwargs),
}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(series=train_series_sample, max_samples_per_ts=MAX_SAMPLES_PER_TS)

# 7. Predictions

forecasts = {}  

for name, model in models.items():
    print(f"\nForecasting con {name}...")
    forecast_scaled = model.predict(n=HORIZON, series=all_series_scaled)
    forecast_original = scaler.inverse_transform(forecast_scaled)
    forecasts[name] = {
        sid: ts.values().flatten()
        for sid, ts in zip(series_ids_order, forecast_original)
    }

print("\nForecast generati per tutti i modelli.")

# 8. Evaluation

all_scores = {name: {'smape': [], 'mase': []} for name in forecasts}

for name, series_forecasts in forecasts.items():
    for sid, forecast in tqdm(series_forecasts.items(), desc=f"Evaluating {name}"):
        idx = id_to_idx[sid]
        train_series_full = train_df.iloc[idx, 1:].dropna().values
        test_series = test_df.iloc[idx, 1:].dropna().values

        h = min(len(test_series), len(forecast))
        y_true = test_series[:h]
        y_pred = forecast[:h]

        s = smape(y_true, y_pred)
        m = mase(y_true, y_pred, train_series_full, seasonality=1)
        if not np.isnan(s):
            all_scores[name]['smape'].append(s)
        if not np.isnan(m):
            all_scores[name]['mase'].append(m)

print("\n" + "="*80)
print(" DEEP LEARNING (darts) RESULTS")
print("="*80)
for name, s in all_scores.items():
    print(f"\n{name}:")
    print(f"   sMAPE: {np.mean(s['smape']):.4f}")
    print(f"   MASE:  {np.mean(s['mase']):.4f}")
    

In [ ]:
# --- canonical calib/test split 
CONTEXT_LENGTH = 90  
MIN_SERIES_LENGTH = CONTEXT_LENGTH + HORIZON + 2  # 106
N_SERIES_CALIB, N_SERIES_TEST = 500, 1000

lengths = train_df.iloc[:, 1:].notna().sum(axis=1)
filtered_ids = train_df.iloc[:, 0][lengths >= MIN_SERIES_LENGTH].tolist()
calib_ids = filtered_ids[:N_SERIES_CALIB]
test_ids = filtered_ids[N_SERIES_CALIB:N_SERIES_CALIB + N_SERIES_TEST]
exclude_ids = set(calib_ids) | set(test_ids)

# --- per-series scale stats: 
scale_stats = {}
for idx in range(len(train_df)):
    sid = train_df.iloc[idx, 0]
    vals = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
    m, s = float(vals.mean()), float(vals.std())
    if s == 0 or np.isnan(s):
        s = 1.0
    scale_stats[sid] = (m, s)

# --- training pool
clean_pool_ids = [sid for sid in train_df.iloc[:, 0].tolist() if sid not in exclude_ids]

clean_series = []
for sid in tqdm(clean_pool_ids, desc="Building clean pool"):
    idx = id_to_idx[sid]
    vals = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
    m, s = scale_stats[sid]
    clean_series.append(TimeSeries.from_values((vals - m) / s))

rng_cal = np.random.RandomState(RANDOM_STATE)
clean_sample_idx = rng_cal.choice(
    len(clean_series), size=min(TRAIN_SERIES_SAMPLE, len(clean_series)), replace=False
)
clean_train_sample = [clean_series[i] for i in clean_sample_idx]
print(f"Training pool pulito: {len(clean_series)} serie disponibili, "
      f"{len(clean_train_sample)} usate per il fit")

# --- refit 
models_cal = {
    "NHiTS": NHiTSModel(**common_kwargs),
    "DLinear": DLinearModel(**common_kwargs),
    "TiDE": TiDEModel(**common_kwargs),
}

for name, model in models_cal.items():
    print(f"\nTraining {name} (pool pulito)...")
    model.fit(series=clean_train_sample, max_samples_per_ts=MAX_SAMPLES_PER_TS)

# --- build context
def build_context_series(ids, holdout):
    series_list = []
    for sid in ids:
        idx = id_to_idx[sid]
        vals = train_df.iloc[idx, 1:].dropna().values.astype(np.float32)
        if holdout:
            vals = vals[:-HORIZON]
        vals = vals[-CONTEXT_LENGTH:]
        m, s = scale_stats[sid]
        series_list.append(TimeSeries.from_values((vals - m) / s))
    return series_list

calib_context = build_context_series(calib_ids, holdout=True)
test_context = build_context_series(test_ids, holdout=False)

# --- forecast + export per ogni modello ---
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

def export_forecasts(forecasts_dict, ids, model_name, split):
    rows = [{"unique_id": uid, **{f"h{i+1}": v for i, v in enumerate(forecasts_dict[uid])}}
            for uid in ids]
    pd.DataFrame(rows).to_csv(RESULTS_DIR / f"{split}_{model_name}.csv", index=False)

for name, model in models_cal.items():
    print(f"\nForecasting {name} su calib/test...")
    calib_pred_scaled = model.predict(n=HORIZON, series=calib_context)
    test_pred_scaled = model.predict(n=HORIZON, series=test_context)

    calib_forecasts = {
        sid: ts.values().flatten() * scale_stats[sid][1] + scale_stats[sid][0]
        for sid, ts in zip(calib_ids, calib_pred_scaled)
    }
    test_forecasts = {
        sid: ts.values().flatten() * scale_stats[sid][1] + scale_stats[sid][0]
        for sid, ts in zip(test_ids, test_pred_scaled)
    }

    export_forecasts(calib_forecasts, calib_ids, name, "calib")
    export_forecasts(test_forecasts, test_ids, name, "test")

print("Exported:", list(models_cal.keys()))